# HU da aorta e limiares cardíacos: MM-WHS × ImageCAS

Comparação descritiva de CTs de **treino**. MM-WHS usa a aorta ascendente rotulada (820) e a união dos sete labels cardíacos. ImageCAS usa aorta **prevista**, escolhida entre exames classificados visualmente como bons no run revisado.

Os HU são medidos no CT original, sem downscale das intensidades. A máscara prevista é restaurada por vizinho mais próximo. Diferenças de aquisição, cobertura anatômica e origem das máscaras limitam a comparação: esta análise não mede Dice entre bancos nem valida clinicamente um limiar. A máscara ImageCAS vem de um pipeline que já usa threshold; sua distribuição pode refletir esse viés de seleção, mesmo com boa avaliação visual.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from utils.project.analysis.aorta_hu_analysis import (
    HU_REGION_NAMES,
    derive_hu_intervals,
    load_imagecas_aorta_hu,
    load_mmwhs_hu_reference,
    select_reviewed_imagecas_exams,
    select_hu_example_records,
    summarize_hu_reference,
    summarize_hu_retention,
)
from utils.project.datasets.ccta import discover_ccta_dataset
from utils.project.dataframe import require_series_column
from utils.project.analysis.mmwhs_heart_analysis import (
    aggregate_mmwhs_heart_hu,
    select_mmwhs_heart_exams,
)
from utils.project.runtime.notebook_env import (
    resolve_existing_path,
    resolve_imagecas_base_path,
)
from utils.visualization.images.intensity import plot_binned_intensity_histogram

### Funções de apresentação
Os helpers abaixo organizam os gráficos específicos desta análise.

In [ ]:
from collections.abc import Mapping
from matplotlib.figure import Figure
from utils.project.analysis.aorta_hu_analysis import HuReferenceVolume, hu_interval_mask
from utils.visualization.images.hu_threshold import hu_axial_mip, reference_axial_slices


def plot_hu_threshold_examples(
    volume: HuReferenceVolume,
    intervals: Mapping[str, tuple[float, float]],
    *,
    hu_window: tuple[float, float] = (-200, 1000),
) -> Figure:
    """Compara CT original e limiares em MIP e três fatias na orientação RAS."""
    if not np.isfinite(hu_window).all() or hu_window[0] >= hu_window[1]:
        raise ValueError("A janela HU deve ter limites finitos e crescentes.")
    reference = volume.regions[
        "mmwhs_heart" if volume.dataset == "MM-WHS" else "imagecas_aorta"
    ]
    if reference.shape != volume.image.shape:
        raise ValueError("Imagem e referência possuem shapes incompatíveis.")
    indices = reference_axial_slices(reference)
    columns: list[tuple[str, tuple[float, float] | None]] = [("CT original", None)]
    columns.extend(
        (f"{HU_REGION_NAMES[key]}\n{low:.1f}–{high:.1f} HU", (low, high))
        for key, (low, high) in intervals.items()
    )
    fig = plt.figure(figsize=(4.1 * len(columns), 12), layout="constrained")
    extent = (
        -volume.spacing[0] / 2,
        (volume.image.shape[0] - 0.5) * volume.spacing[0],
        -volume.spacing[1] / 2,
        (volume.image.shape[1] - 0.5) * volume.spacing[1],
    )
    for column, (name, interval) in enumerate(columns):
        mip = hu_axial_mip(volume.image, interval)
        for row in range(4):
            ax = fig.add_subplot(4, len(columns), row * len(columns) + column + 1)
            if row == 0:
                values = mip
            else:
                values = volume.image[:, :, indices[row - 1]]
                selected = (
                    np.isfinite(values)
                    if interval is None
                    else hu_interval_mask(values, interval)
                )
                values = np.where(selected, values, np.nan)
            ax.set_facecolor("black")
            ax.imshow(
                np.ma.masked_invalid(values.T),
                cmap="gray",
                origin="lower",
                vmin=hu_window[0],
                vmax=hu_window[1],
                extent=extent,
                aspect="equal",
                interpolation="nearest",
            )
            if row == 0:
                ax.set_title(name, fontsize=10)
            if column == 0:
                ax.set_ylabel("MIP axial" if row == 0 else f"Fatia {indices[row - 1]}")
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle(
        f"{volume.dataset} · {volume.exam_id} · HU nativos · RAS\n"
        "Intervalos no volume completo; sem LCC ou segmentação adicional",
        fontsize=12,
    )
    return fig

## 1. Configuração e seleção

IDs explícitos substituem o sorteio; quantidade `None` seleciona todos os elegíveis. O ImageCAS usa apenas a revisão de treino ligada ao snapshot indicado, não uma classificação automática nem outro run. Não são executados vesselness, óstios ou artérias.

A reconstrução precisa reproduzir as contagens persistidas de voxels, círculos e fatias. Essa concordância não prova igualdade voxel a voxel, pois a máscara histórica não foi salva.

In [ ]:
IMAGECAS_PATH = resolve_imagecas_base_path()
MMWHS_PATH = resolve_existing_path(
    "MMWHS_BASE_PATH",
    [IMAGECAS_PATH.parents[1] / "MM-WHS-2017-Dataset"],
    "MM-WHS 2017",
)
N_MMWHS: int | None = 10
N_IMAGECAS: int | None = 5
RANDOM_SEED = 42
MMWHS_EXAM_IDS: list[str] | None = None
IMAGECAS_EXAM_IDS: list[int] | None = None
IMAGECAS_REVIEW_VARIANT = "filter_envelope_current"

LOWER_PERCENTILE = 5.0
UPPER_PERCENTILE = 95.0
HU_BIN_EDGES = np.arange(-1500.0, 3001.0, 10.0)
HU_INTERVAL_OVERRIDES: dict[str, tuple[float, float]] = {}
# Chaves disponíveis: mmwhs_heart, mmwhs_aorta, imagecas_aorta.
N_EXAMPLES = 2
MMWHS_EXAMPLE_IDS: list[str] | None = None
IMAGECAS_EXAMPLE_IDS: list[int] | None = None
HU_WINDOW = (-200.0, 1000.0)

mmwhs_exams = select_mmwhs_heart_exams(
    discover_ccta_dataset("mmwhs", MMWHS_PATH),
    n_exams=N_MMWHS,
    random_seed=RANDOM_SEED,
    exam_ids=MMWHS_EXAM_IDS,
)
imagecas_run, imagecas_exams = select_reviewed_imagecas_exams(
    REPO_ROOT,
    variant=IMAGECAS_REVIEW_VARIANT,
    n_exams=N_IMAGECAS,
    random_seed=RANDOM_SEED,
    exam_ids=IMAGECAS_EXAM_IDS,
)
print(f"MM-WHS: {len(mmwhs_exams)} exames | ImageCAS: {len(imagecas_exams)} exames")
print(f"Run ImageCAS: {imagecas_run.relative_to(REPO_ROOT)}")
display(
    pd.DataFrame(
        {
            "Banco": ["MM-WHS", "ImageCAS"],
            "Exames": [
                ", ".join(require_series_column(mmwhs_exams, "exam_id").astype(str)),
                ", ".join(require_series_column(imagecas_exams, "IMG_ID").astype(str)),
            ],
            "Máscara": [
                "Label anatômico de referência",
                "Predição com boa avaliação visual",
            ],
        }
    )
)

## 2. Distribuições e estatísticas HU

Cada exame contribui com o mesmo peso. As estatísticas incluem todos os HU finitos; valores fora dos bins continuam no denominador das densidades. Regiões ausentes não contribuem como HU zero.

A tabela mostra **média das médias HU**, **desvio entre médias de exames** e **mediana das medianas**, não estatísticas de volumes concatenados. Os mesmos critérios valem para a união cardíaca, calculada a partir de seus voxels reais.

In [ ]:
statistic_parts: list[pd.DataFrame] = []
histogram_parts: list[pd.DataFrame] = []

for _, record in mmwhs_exams.iterrows():
    print(f"HU MM-WHS: {record['exam_id']}")
    volume = load_mmwhs_hu_reference(record)
    case_statistics, case_histogram = summarize_hu_reference(
        volume,
        percentiles=(LOWER_PERCENTILE, UPPER_PERCENTILE),
        bin_edges=HU_BIN_EDGES,
    )
    statistic_parts.append(case_statistics)
    histogram_parts.append(case_histogram)
    del volume

for _, record in imagecas_exams.iterrows():
    print(f"Reconstrução e HU ImageCAS: {record['IMG_ID']}")
    volume = load_imagecas_aorta_hu(IMAGECAS_PATH, imagecas_run, record)
    case_statistics, case_histogram = summarize_hu_reference(
        volume,
        percentiles=(LOWER_PERCENTILE, UPPER_PERCENTILE),
        bin_edges=HU_BIN_EDGES,
    )
    statistic_parts.append(case_statistics)
    histogram_parts.append(case_histogram)
    del volume

statistics = pd.concat(statistic_parts, ignore_index=True)
histograms = pd.concat(histogram_parts, ignore_index=True)
summary, densities = aggregate_mmwhs_heart_hu(statistics, histograms)
display(
    summary.loc[
        :,
        [
            "region_name",
            "exam_count",
            "selected_exam_count",
            "mean_of_means_hu",
            "std_between_exam_means_hu",
            "median_of_medians_hu",
            "mean_outside_range_percent",
            "nonfinite_voxel_count",
        ],
    ]
    .rename(
        columns={
            "region_name": "Região",
            "exam_count": "Exames válidos",
            "selected_exam_count": "Exames selecionados",
            "mean_of_means_hu": "Média das médias HU",
            "std_between_exam_means_hu": "Desvio entre médias HU",
            "median_of_medians_hu": "Mediana das medianas HU",
            "mean_outside_range_percent": "Fora dos bins (%)",
            "nonfinite_voxel_count": "Voxels não finitos",
        }
    )
    .round(2)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
for ax, region, color in zip(
    axes, ("mmwhs_aorta", "imagecas_aorta"), ("#e68a00", "#31688e"), strict=True
):
    region_density = densities.loc[
        require_series_column(densities, "region").eq(region)
    ]
    row = summary.loc[require_series_column(summary, "region").eq(region)].iloc[0]
    plot_binned_intensity_histogram(
        ax,
        region_density,
        value_column="density",
        mean_hu=float(row["mean_of_means_hu"]),
        median_hu=float(row["median_of_medians_hu"]),
        distribution_label="Média das densidades por exame",
        color=color,
    )
    ax.set(title=HU_REGION_NAMES[region], xlabel="Intensidade (HU)", xlim=(-200, 1200))
    ax.legend(fontsize=8)
axes[0].set_ylabel("Densidade média por exame")
fig.tight_layout()
plt.show()
plt.close(fig)

## 3. Intervalos derivados

O padrão é P5–P95. Cada limite geral é a **média dos percentis por exame**, mantendo peso igual; não é o percentil de todos os voxels concatenados. Essa faixa média não garante preservar 90% de cada região/exame. `HU_INTERVAL_OVERRIDES` substitui os limites por valores absolutos.

Os três intervalos são aplicados a ambos os bancos. A faixa cardíaca do MM-WHS também é aplicada ao ImageCAS, mas não há label cardíaco nesse banco para medir sua cobertura.

In [ ]:
intervals = derive_hu_intervals(statistics, overrides=HU_INTERVAL_OVERRIDES)
interval_table = pd.DataFrame(
    [
        {
            "Origem": HU_REGION_NAMES[region],
            "Exames válidos": int(
                summary.loc[
                    require_series_column(summary, "region").eq(region), "exam_count"
                ].iloc[0]
            ),
            "Limite inferior HU": low,
            "Limite superior HU": high,
            "Critério": "Valores absolutos"
            if region in HU_INTERVAL_OVERRIDES
            else f"Média de P{LOWER_PERCENTILE:g} e P{UPPER_PERCENTILE:g} por exame",
        }
        for region, (low, high) in intervals.items()
    ]
)
display(interval_table.round(2))
display(
    statistics.loc[
        :,
        [
            "dataset",
            "exam_id",
            "region_name",
            "volume_ml",
            "mean_hu",
            "lower_percentile_hu",
            "upper_percentile_hu",
        ],
    ]
    .rename(
        columns={
            "dataset": "Banco",
            "exam_id": "Exame",
            "region_name": "Região",
            "volume_ml": "Volume (ml)",
            "mean_hu": "Média HU",
            "lower_percentile_hu": f"P{LOWER_PERCENTILE:g} HU",
            "upper_percentile_hu": f"P{UPPER_PERCENTILE:g} HU",
        }
    )
    .round(2)
)

## 4. MIP, fatias e preservação das regiões

Cada figura mostra CT original e os três intervalos, usando o volume completo. O MIP é calculado **após selecionar os HU**. Pixels sem candidatos ficam mascarados; voxels rejeitados não são substituídos por zero.

As três fatias ficam em 25%, 50% e 75% da extensão axial do coração MM-WHS ou da aorta prevista ImageCAS. As imagens estão em orientação canônica RAS, com proporção física e janela HU iguais entre versões.

As tabelas descrevem somente os exames exibidos: fração dos HU finitos selecionada no volume e fração preservada em cada região disponível. Preservação da aorta prevista não equivale a sensibilidade contra ground truth. Outros tecidos podem ter HU semelhantes; não há LCC ou nova segmentação.

In [ ]:
def show_retention(frame: pd.DataFrame) -> None:
    """Mostra seleção global e preservação da região nos exemplos."""
    table = frame.loc[
        :,
        [
            "dataset",
            "exam_id",
            "interval",
            "region",
            "selected_volume_percent",
            "retained_region_percent",
        ],
    ].copy()
    table["interval"] = require_series_column(table, "interval").map(
        lambda key: HU_REGION_NAMES[str(key)]
    )
    table["region"] = require_series_column(table, "region").map(
        lambda key: HU_REGION_NAMES[str(key)]
    )
    display(
        table.rename(
            columns={
                "dataset": "Banco",
                "exam_id": "Exame",
                "interval": "Intervalo",
                "region": "Região avaliada",
                "selected_volume_percent": "Volume selecionado (%)",
                "retained_region_percent": "Região preservada (%)",
            }
        ).round(2)
    )

In [ ]:
mmwhs_retention: list[pd.DataFrame] = []
for _, record in select_hu_example_records(
    mmwhs_exams, "exam_id", requested=MMWHS_EXAMPLE_IDS, n_examples=N_EXAMPLES
).iterrows():
    volume = load_mmwhs_hu_reference(record)
    mmwhs_retention.append(summarize_hu_retention(volume, intervals))
    fig = plot_hu_threshold_examples(volume, intervals, hu_window=HU_WINDOW)
    plt.show()
    plt.close(fig)
    del volume
show_retention(pd.concat(mmwhs_retention, ignore_index=True))

In [ ]:
imagecas_retention: list[pd.DataFrame] = []
for _, record in select_hu_example_records(
    imagecas_exams, "IMG_ID", requested=IMAGECAS_EXAMPLE_IDS, n_examples=N_EXAMPLES
).iterrows():
    volume = load_imagecas_aorta_hu(IMAGECAS_PATH, imagecas_run, record)
    imagecas_retention.append(summarize_hu_retention(volume, intervals))
    fig = plot_hu_threshold_examples(volume, intervals, hu_window=HU_WINDOW)
    plt.show()
    plt.close(fig)
    del volume
show_retention(pd.concat(imagecas_retention, ignore_index=True))